In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv


In [4]:
import csv
import json
import re
import sys
from datetime import timezone
from email.utils import parsedate_to_datetime
from typing import Dict, Iterable, Iterator, List, Optional, Tuple
import pandas as pd

# ============================================================
# CSV Field Size Limit (handle large email bodies)
# ============================================================

def set_csv_field_size_limit() -> None:
    """Increase CSV field size limit to handle large email bodies."""
    limit = sys.maxsize
    while True:
        try:
            csv.field_size_limit(limit)
            return
        except OverflowError:
            limit //= 10

set_csv_field_size_limit()


# ============================================================
# Email Parsing Functions
# ============================================================

def split_headers_and_body(message: str) -> Tuple[str, str]:
    """Split email into headers and body."""
    normalized = message.replace("\r\n", "\n")
    if "\n\n" not in normalized:
        return normalized, ""
    return normalized.split("\n\n", 1)


def parse_headers(header_text: str) -> Dict[str, str]:
    """Parse email headers into a dictionary."""
    headers: Dict[str, str] = {}
    current_name = None

    for raw_line in header_text.split("\n"):
        if not raw_line:
            continue
        # Handle multi-line headers (e.g., long Subject)
        if raw_line[0].isspace() and current_name:
            headers[current_name] = f"{headers[current_name]} {raw_line.strip()}"
            continue
        if ":" not in raw_line:
            continue
        name, value = raw_line.split(":", 1)
        current_name = name.strip()
        headers[current_name] = value.strip()

    return headers


def normalize_date(date_raw: str) -> Optional[str]:
    """Convert email date to UTC format."""
    if not date_raw:
        return None

    # Remove timezone abbreviation like (PDT)
    cleaned = re.sub(r"\s+\([^)]+\)$", "", date_raw.strip())
    try:
        parsed = parsedate_to_datetime(cleaned)
    except (TypeError, ValueError):
        return None

    # Assume UTC if no timezone info
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)

    return parsed.astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def normalize_cell(value) -> str:
    """Convert None/NaN to empty string."""
    if value is None:
        return ""
    try:
        if value != value:  # NaN check
            return ""
    except Exception:
        pass
    return str(value)


def sanitize_row(row: Dict) -> Dict[str, str]:
    """Convert all values in a row to strings."""
    return {str(key): normalize_cell(value) for key, value in row.items()}


def build_record(row: Dict) -> Dict:
    """
    Convert a raw CSV row into a structured email record.
    
    Args:
        row: Dict with 'file' and 'message' keys
        
    Returns:
        Structured email record with parsed fields
    """
    normalized_row = sanitize_row(row)
    raw_message = normalized_row.get("message", "")
    
    # Split headers and body
    header_text, body = split_headers_and_body(raw_message)
    headers = parse_headers(header_text)
    clean_body = body.strip()
    
    return {
        "file_path": normalized_row.get("file"),
        "message_id": headers.get("Message-ID"),
        "sent_at": normalize_date(headers.get("Date", "")),
        "sender": headers.get("From"),
        "to": headers.get("To"),
        "cc": headers.get("Cc") or headers.get("X-cc"),
        "bcc": headers.get("Bcc") or headers.get("X-bcc"),
        "subject": headers.get("Subject"),
        "x_folder": headers.get("X-Folder"),
        "x_origin": headers.get("X-Origin"),
        "x_filename": headers.get("X-FileName"),
        "body": clean_body,
        "body_length": len(clean_body),
    }


# ============================================================
# Batch Processing Functions
# ============================================================

def iter_records_from_dataframe(df: pd.DataFrame) -> Iterator[Dict]:
    """Iterate over DataFrame rows and yield parsed records."""
    for _, row in df.iterrows():
        yield build_record(row.to_dict())


def batch_records(
    records: Iterable[Dict],
    batch_size: int,
    max_records: Optional[int] = None
) -> Iterator[List[Dict]]:
    """Group records into batches."""
    batch: List[Dict] = []
    emitted = 0

    for record in records:
        batch.append(record)
        emitted += 1

        if len(batch) >= batch_size:
            yield batch
            batch = []

        if max_records is not None and emitted >= max_records:
            break

    if batch:
        yield batch


In [6]:
# ============================================================
# Main Processing
# ============================================================

def main():
    # Kaggle dataset path (auto-mounted when dataset is added)
    DATA_PATH = "/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv"
    
    # Configuration
    BATCH_SIZE = 5000  # Records per output file
    MAX_RECORDS = None  # Set to None for all records, or number for testing
    
    print("=" * 60)
    print("Enron Email Parser")
    print("=" * 60)
    print(f"Reading from: {DATA_PATH}")
    
    # Read CSV
    df = pd.read_csv(DATA_PATH, nrows=MAX_RECORDS if MAX_RECORDS else None)
    print(f"Loaded {len(df)} rows")
    
    # Process and save batches
    output_dir = "/kaggle/working/parsed_emails"
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Writing parsed batches to: {output_dir}")
    
    batch_count = 0
    total_records = 0
    
    for batch in batch_records(
        iter_records_from_dataframe(df),
        BATCH_SIZE,
        MAX_RECORDS
    ):
        # Write batch to JSON file
        batch_path = os.path.join(output_dir, f"batch_{batch_count:06d}.json")
        with open(batch_path, 'w', encoding='utf-8') as f:
            json.dump(batch, f, ensure_ascii=False, indent=2)
        
        total_records += len(batch)
        print(f"Wrote {len(batch)} records -> batch_{batch_count:06d}.json")
        batch_count += 1
    
    print(f"\n Processed {total_records} records into {batch_count} files")
    print(f"Output directory: {output_dir}")
    
    # Quick statistics
    print("\n" + "=" * 60)
    print("Quick Statistics (first batch only)")
    print("=" * 60)
    
    first_batch_path = os.path.join(output_dir, "batch_000000.json")
    if os.path.exists(first_batch_path):
        with open(first_batch_path, 'r') as f:
            first_batch = json.load(f)
        
        print(f"Sample record keys: {list(first_batch[0].keys())}")
        print(f"\nSample record:")
        print(json.dumps(first_batch[0], indent=2, ensure_ascii=False)[:1000] + "...")


if __name__ == "__main__":
    main()

Enron Email Parser
Reading from: /kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv
Loaded 517401 rows
Writing parsed batches to: /kaggle/working/parsed_emails
Wrote 5000 records -> batch_000000.json
Wrote 5000 records -> batch_000001.json
Wrote 5000 records -> batch_000002.json
Wrote 5000 records -> batch_000003.json
Wrote 5000 records -> batch_000004.json
Wrote 5000 records -> batch_000005.json
Wrote 5000 records -> batch_000006.json
Wrote 5000 records -> batch_000007.json
Wrote 5000 records -> batch_000008.json
Wrote 5000 records -> batch_000009.json
Wrote 5000 records -> batch_000010.json
Wrote 5000 records -> batch_000011.json
Wrote 5000 records -> batch_000012.json
Wrote 5000 records -> batch_000013.json
Wrote 5000 records -> batch_000014.json
Wrote 5000 records -> batch_000015.json
Wrote 5000 records -> batch_000016.json
Wrote 5000 records -> batch_000017.json
Wrote 5000 records -> batch_000018.json
Wrote 5000 records -> batch_000019.json
Wrote 5000 records -> batc

In [7]:
!pip install confluent_kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 48.2 MB/s eta 0:00:0000:0100:01


In [26]:
"""
Enron Email Kafka Producer - Clean version for new topic
"""

import json
import os
from confluent_kafka import Producer

# ============================================================
# Kafka Configuration
# ============================================================

conf = {
    "bootstrap.servers": "pkc-ox31np.ap-southeast-7.aws.confluent.cloud:9092",
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": "S4AU73FQKMOAMSPL",
    "sasl.password": "cfltEuKzwF9VDz4Y2XwilhMpyc+9rxUP4xYfpOFYbbC1VefzzNc8rOYHfIavyVLw",
    "client.id": "enron-producer",
    "session.timeout.ms": 45000,
    "message.max.bytes": 8388608  # 8 MB
}

producer = Producer(conf)

# NEW topic name
topic = "raw_emails_topic"

# ============================================================
# Checkpoint (start fresh, ignore old checkpoint)
# ============================================================

CHECKPOINT_FILE = "/kaggle/working/kafka_checkpoint_new.json"  # New file

def read_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
            return data.get('file_idx', 0), data.get('record_idx', 0)
    return 0, 0

def write_checkpoint(file_idx, record_idx):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'file_idx': file_idx, 'record_idx': record_idx}, f)

def delivery_report(err, msg):
    if err:
        print(f"Delivery failed: {err}")

# ============================================================
# Send emails
# ============================================================

def send_emails_to_kafka(
    input_dir="/kaggle/working/parsed_emails",
    max_files=None,
    max_records_per_file=None
):
    all_files = sorted([f for f in os.listdir(input_dir) if f.startswith("batch_")])
    if max_files:
        all_files = all_files[:max_files]
    
    start_file_idx, start_record_idx = read_checkpoint()
    
    print(f"Topic: {topic}")
    print(f"Total files: {len(all_files)} | Resuming from file {start_file_idx}, record {start_record_idx}")
    
    total_sent = 0
    pending = 0
    skipped = 0
    
    for file_idx, file_name in enumerate(all_files):
        if file_idx < start_file_idx:
            continue
        
        with open(os.path.join(input_dir, file_name), 'r') as f:
            emails = json.load(f)
        
        start = start_record_idx if file_idx == start_file_idx else 0
        end = len(emails)
        if max_records_per_file:
            end = min(start + max_records_per_file, end)
        
        for record_idx in range(start, end):
            email = emails[record_idx]
            key = email.get('file_path', file_name)
            value = json.dumps(email, ensure_ascii=False).encode('utf-8')
            
            # Skip messages larger than 8 MB
            if len(value) > 8388608:
                print(f"Skipping: {key} ({len(value) / 1024 / 1024:.2f} MB)")
                skipped += 1
                continue
            
            producer.produce(topic, key=key, value=value, callback=delivery_report)
            total_sent += 1
            pending += 1
            
            if pending >= 500:
                producer.flush()
                write_checkpoint(file_idx, record_idx + 1)
                pending = 0
        
        if pending > 0:
            producer.flush()
            pending = 0
        
        write_checkpoint(file_idx + 1, 0)
        print(f"✅ Sent {file_name} ({end - start} records) | Total: {total_sent} | Skipped: {skipped}")
    
    producer.flush()
    write_checkpoint(len(all_files), 0)
    print(f"\n🎉 All done! Total sent: {total_sent} | Skipped: {skipped}")


%4|1775372373.252|CONFWARN|enron-producer#producer-9| [thrd:app]: Configuration property session.timeout.ms is a consumer property and will be ignored by this producer instance
%6|1775372376.607|GETSUBSCRIPTIONS|enron-producer#producer-9| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to Ul//PgSFQguZrZ3rTBZJVg


In [27]:
if __name__ == "__main__":
    send_emails_to_kafka()

Topic: raw_emails_topic
Total files: 104 | Resuming from file 0, record 0
✅ Sent batch_000000.json (5000 records) | Total: 5000 | Skipped: 0
✅ Sent batch_000001.json (5000 records) | Total: 10000 | Skipped: 0
✅ Sent batch_000002.json (5000 records) | Total: 15000 | Skipped: 0
✅ Sent batch_000003.json (5000 records) | Total: 20000 | Skipped: 0
✅ Sent batch_000004.json (5000 records) | Total: 25000 | Skipped: 0
✅ Sent batch_000005.json (5000 records) | Total: 30000 | Skipped: 0
✅ Sent batch_000006.json (5000 records) | Total: 35000 | Skipped: 0
✅ Sent batch_000007.json (5000 records) | Total: 40000 | Skipped: 0
✅ Sent batch_000008.json (5000 records) | Total: 45000 | Skipped: 0
✅ Sent batch_000009.json (5000 records) | Total: 50000 | Skipped: 0
✅ Sent batch_000010.json (5000 records) | Total: 55000 | Skipped: 0
✅ Sent batch_000011.json (5000 records) | Total: 60000 | Skipped: 0
✅ Sent batch_000012.json (5000 records) | Total: 65000 | Skipped: 0
✅ Sent batch_000013.json (5000 records) | T